# Tema: Historial, time travel y RESTORE

## Objetivos
Crear versiones 0–4, consultar snapshots y restaurar sin borrar el historial.

## Conceptos importantes para el examen
VERSION AS OF; TIMESTAMP AS OF; RESTORE genera otro commit; VACUUM limita recuperación histórica.

**Dificultad:** Intermedio · **Tiempo estimado:** 65 min.



Ejecuta la preparación una vez; después avanza celda a celda. Las soluciones modifican datos: úsalas tras tu intento. Para volver al estado inicial, ejecuta de nuevo la preparación completa (crea otro schema). No uses «Run all» para estudiar.

- [ ] Completado
- [ ] Necesito repasar
- [ ] Dominado

## Preparación y datos ficticios
Se necesita un notebook Python en Databricks con Spark y Unity Catalog. Solo se crean objetos en el schema de prácticas mostrado.

In [ ]:
# Cada ejecución de esta celda crea un schema NUEVO y aislado.
# El catálogo debe existir y permitir USE CATALOG y CREATE SCHEMA.
# Si no puedes crear schemas, pide uno de prácticas exclusivo y cambia SCHEMA.
import re
import uuid
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.window import Window

dbutils.widgets.text("catalog", spark.sql("SELECT current_catalog()").first()[0])
CATALOG = dbutils.widgets.get("catalog")
RUN_ID = uuid.uuid4().hex[:10]
SCHEMA = "dea_07_" + RUN_ID
def ident(value):
    return "`" + value.replace("`", "``") + "`"
spark.sql(f"USE CATALOG {ident(CATALOG)}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {ident(SCHEMA)}")
spark.sql(f"USE SCHEMA {ident(SCHEMA)}")
spark.sql("SET TIME ZONE 'UTC'")
print(f"Objetos de esta sesión: {CATALOG}.{SCHEMA}")
# No se borran automáticamente schemas, tablas ni checkpoints.


In [ ]:
employees = spark.createDataFrame(
    [(i, f"Empleado {i:02d}", ["Data", "Sales", "Finance"][i % 3],
      30000 + i * 1500, i % 4 != 0,
      datetime(2026, 1, 1), datetime(2026, 1, 1)) for i in range(1, 19)],
    "employee_id INT, name STRING, department STRING, salary INT, active BOOLEAN, created_at TIMESTAMP, updated_at TIMESTAMP"
)
employees.createOrReplaceTempView("employees_seed")
employees.write.format("delta").mode("errorifexists").saveAsTable("employees")
display(employees.orderBy("employee_id"))

## PARTE 1 - EJEMPLOS GUIADOS

### 1. Crear versiones observables
La tabla timeline es nueva en este schema: CTAS produce versión 0.

In [ ]:
%sql
CREATE TABLE timeline USING DELTA AS SELECT employee_id, salary FROM employees;
INSERT INTO timeline VALUES (99, 50000);
UPDATE timeline SET salary = 60000 WHERE employee_id = 1;
DELETE FROM timeline WHERE employee_id = 2;
MERGE INTO timeline t USING (SELECT 3 employee_id, 70000 salary) s ON t.employee_id = s.employee_id
WHEN MATCHED THEN UPDATE SET salary = s.salary WHEN NOT MATCHED THEN INSERT *;
DESCRIBE HISTORY timeline;

### 2. Leer versiones sin cambiar el estado actual

In [ ]:
%sql
SELECT 'v0' snapshot, COUNT(*) n FROM timeline VERSION AS OF 0
UNION ALL SELECT 'v1', COUNT(*) FROM timeline VERSION AS OF 1
UNION ALL SELECT 'actual', COUNT(*) FROM timeline;

### 3. Consultar por timestamp exacto de un commit
UTC y timestamp obtenido del historial evitan adivinar la hora.

In [ ]:
history = spark.sql("DESCRIBE HISTORY timeline")
ts_v2 = history.filter("version = 2").select("timestamp").first()[0]
display(spark.sql(f"SELECT * FROM timeline TIMESTAMP AS OF '{ts_v2.isoformat(sep=' ')}'"))

## PARTE 2 - EJERCICIOS
Resuelve todos antes de abrir las soluciones. Los ejercicios se realizan en orden y pueden usar resultados anteriores.

### EJERCICIO 1
Muestra versiones 0, 1, 2, 3 y 4 con operación y timestamp.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 2
Obtén el salario del empleado 1 en versiones 0 y 2.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 3
Consulta mediante TIMESTAMP AS OF la versión del commit de DELETE y comprueba que falta el 2.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 4
Restaura timeline a versión 1. Verifica 19 filas y observa la nueva versión.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 5
Demuestra que versión 4 sigue accesible después de RESTORE; compara salario del 3. Explica la limitación de retención.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


## PARTE 3 - PISTAS
**Pista 1:** Ordena DESCRIBE HISTORY.

**Pista 2:** VERSION AS OF va después del nombre.

**Pista 3:** Extrae el timestamp de versión 3.

**Pista 4:** RESTORE TABLE no rebobina los números de commit.

**Pista 5:** No ejecutes VACUUM con retención cero.

## PARTE 4 - SOLUCIONES
**Detente aquí si todavía estás practicando.** Referencias completas para comparar después de resolver. Puedes plegar esta sección en Databricks.

### Solución 1

In [ ]:
display(spark.sql("DESCRIBE HISTORY timeline").select("version", "timestamp", "operation").orderBy("version"))

### Solución 2

In [ ]:
%sql
SELECT 'v0' version, salary FROM timeline VERSION AS OF 0 WHERE employee_id=1
UNION ALL SELECT 'v2', salary FROM timeline VERSION AS OF 2 WHERE employee_id=1;

### Solución 3

In [ ]:
ts = spark.sql("DESCRIBE HISTORY timeline").filter("version = 3").first().timestamp
snapshot = spark.sql(f"SELECT * FROM timeline TIMESTAMP AS OF '{ts.isoformat(sep=' ')}'")
assert snapshot.filter("employee_id = 2").count() == 0
display(snapshot)

### Solución 4

In [ ]:
%sql
RESTORE TABLE timeline TO VERSION AS OF 1;
SELECT COUNT(*) AS n FROM timeline;
DESCRIBE HISTORY timeline;

### Solución 5

In [ ]:
%sql
SELECT 'v4' snapshot, salary FROM timeline VERSION AS OF 4 WHERE employee_id = 3
UNION ALL SELECT 'restaurada', salary FROM timeline WHERE employee_id = 3;
-- El time travel requiere log y archivos de la versión.
-- VACUUM puede eliminar archivos antiguos aunque conserves entradas de historial.

## PARTE 5 - PREGUNTAS TIPO EXAMEN
Preguntas originales de práctica; no son preguntas oficiales.

### Pregunta 1
Tras restaurar v1 desde v4, ¿qué sucede?

A. Desaparecen v2–v4

B. Se confirma una nueva versión con datos de v1

C. La versión actual pasa a numerarse 1

D. No cambian los datos

### Pregunta 2
¿Qué puede impedir leer una versión antigua?

A. ORDER BY

B. SELECT *

C. Archivos eliminados por retención/VACUUM

D. Un alias SQL

### Pregunta 3
¿Qué consulta histórica modifica el estado actual?

A. RESTORE TABLE

B. SELECT VERSION AS OF

C. DESCRIBE HISTORY

D. SELECT TIMESTAMP AS OF

### Respuestas y explicación
**1. B** — RESTORE registra la restauración como una operación nueva.

**2. C** — El lector necesita los archivos de ese snapshot.

**3. A** — RESTORE es una escritura; las otras son lecturas.

## PARTE 6 - RETO FINAL
Simula una actualización errónea, determina el último estado correcto, compara ambos snapshots y restaura. Documenta las versiones sin depender de números memorizados.

Anota tu decisión, implementa el código y muestra evidencias. No se incluye solución para este reto.

In [ ]:
# TU RETO: código y verificaciones
